# OCBridge / Hiring Copilot
## AI-Executed Candidate Matching & Recommendation Workflow

**Steps:**
1. Cell 1: Install dependencies
2. Cell 2: Set API Key
3. Cell 3a: Upload JD PDF
4. Cell 3b: Upload Resume PDFs
5. Cell 4: Parse JD requirements
6. Cell 5: Evaluate all candidates
7. Cell 6: Display results
8. Cell 7: Export PDF report

In [9]:
# Cell 1: Install dependencies
!pip install anthropic pypdf reportlab -q

In [10]:
# Cell 2: Set API Key
import anthropic
import json

API_KEY = 'paste_your_api_key_here'
client = anthropic.Anthropic(api_key=API_KEY)

def ask(prompt, system=None):
    kwargs = dict(
        model='claude-sonnet-4-6',
        max_tokens=1000,
        temperature=0,
        messages=[{'role': 'user', 'content': prompt}]
    )
    if system:
        kwargs['system'] = system
    msg = client.messages.create(**kwargs)
    return msg.content[0].text

def ask_json(prompt, system=None):
    raw = ask(prompt, system=system)
    # Strip markdown code fences if present
    clean = raw.strip()
    if clean.startswith('```'):
        clean = clean.split('\n', 1)[-1]
        clean = clean.rsplit('```', 1)[0]
    return json.loads(clean.strip())

print('Initialized successfully')


Initialized successfully


In [11]:
# Cell 3a: Upload JD PDF
from google.colab import files
from pypdf import PdfReader
import os

def extract_pdf_text(filename, content):
    with open(filename, 'wb') as f:
        f.write(content)
    reader = PdfReader(filename)
    text = ''
    for page in reader.pages:
        text += page.extract_text() or ''
    text = text.encode('utf-8', errors='ignore').decode('utf-8')
    return text

print('Upload the JD PDF:')
uploaded_jd = files.upload()

jd_text = ''
for filename, content in uploaded_jd.items():
    jd_text = extract_pdf_text(filename, content)
    print(f'JD loaded: {filename} ({len(jd_text)} chars)')


Upload the JD PDF:


Saving Applied_AI_Product_Intern_JD.pdf to Applied_AI_Product_Intern_JD (1).pdf
JD loaded: Applied_AI_Product_Intern_JD (1).pdf (5435 chars)


In [12]:
# Cell 3b: Upload Resume PDFs
print('Upload all resume PDFs (you can select multiple files):')
uploaded_resumes = files.upload()

resumes = {}
for filename, content in uploaded_resumes.items():
    text = extract_pdf_text(filename, content)
    name = os.path.splitext(filename)[0]
    resumes[name] = text
    print(f'Resume loaded: {filename} ({len(text)} chars)')

print(f'\nTotal resumes: {len(resumes)}')


Upload all resume PDFs (you can select multiple files):


Saving Synthetic_Resume_01.pdf to Synthetic_Resume_01 (1).pdf
Saving Synthetic_Resume_02.pdf to Synthetic_Resume_02 (1).pdf
Saving Synthetic_Resume_03.pdf to Synthetic_Resume_03 (1).pdf
Saving Synthetic_Resume_04.pdf to Synthetic_Resume_04 (1).pdf
Saving Synthetic_Resume_05.pdf to Synthetic_Resume_05 (1).pdf
Saving Synthetic_Resume_06.pdf to Synthetic_Resume_06 (1).pdf
Saving Synthetic_Resume_07.pdf to Synthetic_Resume_07 (1).pdf
Resume loaded: Synthetic_Resume_01 (1).pdf (3215 chars)
Resume loaded: Synthetic_Resume_02 (1).pdf (5544 chars)
Resume loaded: Synthetic_Resume_03 (1).pdf (2998 chars)
Resume loaded: Synthetic_Resume_04 (1).pdf (3735 chars)
Resume loaded: Synthetic_Resume_05 (1).pdf (3663 chars)
Resume loaded: Synthetic_Resume_06 (1).pdf (3840 chars)
Resume loaded: Synthetic_Resume_07 (1).pdf (4320 chars)

Total resumes: 7


In [13]:
# Cell 4: Parse JD requirements
jd_parse_prompt = f'''You are a senior recruiting consultant at an AI-native recruiting firm.

Analyze the following job description and extract structured requirements.
Output ONLY valid JSON, no preamble, no markdown backticks.

Job Description:
{jd_text}

Return this exact JSON structure:
{{
  "must_have": ["list of non-negotiable requirements"],
  "strong_signals": ["list of differentiating qualities that significantly boost recommendation"],
  "nice_to_have": ["list of bonus qualifications that slightly boost recommendation"],
  "dealbreakers": ["list of conditions that immediately disqualify a candidate"],
  "work_style": ["list of behavioral and soft skill requirements"]
}}'''

print('Parsing JD requirements...')
jd_requirements = ask_json(jd_parse_prompt)
print('Done.\n')
for key, values in jd_requirements.items():
    print(f'{key.upper()}:')
    for v in values:
        print(f'  - {v}')
    print()


Parsing JD requirements...
Done.

MUST_HAVE:
  - AI-native product thinking ability
  - Ability to translate ambiguous business discussions into structured product features
  - Python programming proficiency
  - Experience with APIs and automation tools
  - Experience with LLM prompting and workflows
  - Familiarity with AI coding tools such as ChatGPT, Claude, or Codex
  - Ability to rapidly prototype and iterate on product ideas
  - Strong business and workflow sensitivity
  - Comfort operating in ambiguous, fast-moving environments
  - Strong written and verbal communication skills with both technical and non-technical stakeholders

STRONG_SIGNALS:
  - Hands-on experience with browser automation
  - Experience building workflow orchestration systems or agents
  - Demonstrated ability to build and ship side projects quickly and independently
  - Experience with FastAPI or similar lightweight backend frameworks
  - Prior exposure to recruiting, HR tech, or operational workflow tools
 

In [14]:
# Cell 5: Evaluate all candidates
results = []

SYSTEM_PROMPT = '''You are a senior recruiting consultant at OCBridge, an AI-native recruiting firm.
You evaluate candidates for technical AI product roles at early-stage startups.
Be specific, evidence-based, and practical in your assessments.
Do not ask clarifying questions. Provide complete output directly.
Always output only valid JSON, no preamble, no markdown backticks.'''

for name, resume_text in resumes.items():
    print(f'Evaluating: {name}...')

    # Prompt 2: Candidate evaluation
    eval_prompt = f'''Evaluate this candidate against the following role requirements.

ROLE REQUIREMENTS:
{json.dumps(jd_requirements, indent=2)}

CANDIDATE RESUME:
{resume_text}

Return this exact JSON structure:
{{
  "candidate_name": "full name extracted from resume",
  "recommendation": "Ready to Submit | Validate First | Do Not Submit",
  "brief_reason": "one sentence explaining the recommendation",
  "main_risk": "one sentence describing the main risk",
  "strengths": ["strength with specific evidence from resume"],
  "risks": ["risk with specific evidence or absence of evidence"]
}}'''

    evaluation = ask_json(eval_prompt, system=SYSTEM_PROMPT)
    recommendation = evaluation['recommendation']
    print(f'  Recommendation: {recommendation}')

    # Prompt 3 & 4: only for non-rejected candidates
    validation_questions = []
    client_summary = ''

    if recommendation != 'Do Not Submit':

        # Prompt 3: Validation questions
        depth = '1-2 lightweight confirmation questions' if recommendation == 'Ready to Submit' else '3-4 in-depth questions targeting specific risks'
        validation_prompt = f'''Based on this candidate evaluation, generate recruiter validation questions.

CANDIDATE: {name}
EVALUATION:
{json.dumps(evaluation, indent=2)}

Generate {depth} a recruiter should ask this candidate before submission.
Questions should target unverified claims, gaps, or risks.
Be specific - reference actual items from the resume.

Return this exact JSON structure:
{{
  "questions": ["question 1", "question 2"]
}}'''

        validation_data = ask_json(validation_prompt, system=SYSTEM_PROMPT)
        validation_questions = validation_data['questions']

        # Prompt 4: Client-ready summary
        summary_prompt = f'''Write a client-ready candidate summary for submission to a hiring manager.

CANDIDATE: {name}
EVALUATION:
{json.dumps(evaluation, indent=2)}

Write 2-3 sentences in professional business language suitable for a client hiring manager.
Focus on what makes this candidate compelling for the role.
Do not use technical jargon. Do not mention scores or internal evaluation details.

Return this exact JSON structure:
{{
  "summary": "2-3 sentence summary here"
}}'''

        summary_data = ask_json(summary_prompt, system=SYSTEM_PROMPT)
        client_summary = summary_data['summary']

    results.append({
        'name': f"{evaluation.get('candidate_name', name)} ({name})",
        'recommendation': recommendation,
        'brief_reason': evaluation['brief_reason'],
        'main_risk': evaluation['main_risk'],
        'strengths': evaluation['strengths'],
        'risks': evaluation['risks'],
        'validation_questions': validation_questions,
        'client_summary': client_summary
    })

    print(f'  Done.')

order = {'Ready to Submit': 0, 'Validate First': 1, 'Do Not Submit': 2}
results.sort(key=lambda x: order.get(x['recommendation'], 1))
print('\nAll candidates evaluated.')


Evaluating: Synthetic_Resume_01 (1)...
  Recommendation: Validate First
  Done.
Evaluating: Synthetic_Resume_02 (1)...
  Recommendation: Validate First
  Done.
Evaluating: Synthetic_Resume_03 (1)...
  Recommendation: Validate First
  Done.
Evaluating: Synthetic_Resume_04 (1)...
  Recommendation: Do Not Submit
  Done.
Evaluating: Synthetic_Resume_05 (1)...
  Recommendation: Validate First
  Done.
Evaluating: Synthetic_Resume_06 (1)...
  Recommendation: Validate First
  Done.
Evaluating: Synthetic_Resume_07 (1)...
  Recommendation: Validate First
  Done.

All candidates evaluated.


In [15]:
# Cell 6: Display results
print('=' * 80)
print('RECOMMENDATION TABLE')
print('=' * 80)
print(f"{'Candidate':<35} {'Recommendation':<20} {'Brief Reason'}")
print('-' * 100)
for r in results:
    print(f"{r['name']:<35} {r['recommendation']:<20} {r['brief_reason'][:50]}")
print('=' * 80)

print('\n\nDETAILED REPORTS')
for r in results:
    print('\n' + '=' * 80)
    print(f"CANDIDATE: {r['name']}")
    print(f"RECOMMENDATION: {r['recommendation']}")
    print(f"BRIEF REASON: {r['brief_reason']}")
    print(f"MAIN RISK: {r['main_risk']}")
    print('\nSTRENGTHS:')
    for s in r['strengths']:
        print(f'  - {s}')
    print('\nRISKS:')
    for s in r['risks']:
        print(f'  - {s}')
    if r['validation_questions']:
        print('\nVALIDATION QUESTIONS:')
        for i, q in enumerate(r['validation_questions'], 1):
            print(f'  {i}. {q}')
    if r['client_summary']:
        print(f'\nCLIENT SUMMARY:\n  {r["client_summary"]}')


RECOMMENDATION TABLE
Candidate                           Recommendation       Brief Reason
----------------------------------------------------------------------------------------------------
Alex Pemberton (Synthetic_Resume_01 (1)) Validate First       Alex shows strong entrepreneurial drive and produc
Chen Yuxuan (Synthetic_Resume_02 (1)) Validate First       Chen shows solid Python proficiency and real LLM/a
Jordan McCallister (Synthetic_Resume_03 (1)) Validate First       Jordan shows strong technical foundations and rele
Priya Raghunathan (Synthetic_Resume_05 (1)) Validate First       Priya shows genuine technical initiative and AI pr
Zhang Ruoxi (Synthetic_Resume_06 (1)) Validate First       Zhang shows strong AI product thinking and Python 
Ryan Chen (Synthetic_Resume_07 (1)) Validate First       Ryan shows strong AI product and engineering depth
Li Mengqi (Synthetic_Resume_04 (1)) Do Not Submit        Li Mengqi lacks the core AI-native product thinkin


DETAILED REPORTS

CANDID

In [16]:
# Cell 7: Export PDF report
from reportlab.lib.pagesizes import A4
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, HRFlowable, Table, TableStyle
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib import colors
from reportlab.lib.units import mm
from google.colab import files
import datetime

output_filename = 'ocbridge_candidate_report.pdf'
doc = SimpleDocTemplate(output_filename, pagesize=A4,
    rightMargin=20*mm, leftMargin=20*mm, topMargin=20*mm, bottomMargin=20*mm)

styles = getSampleStyleSheet()
s_title = ParagraphStyle('T', parent=styles['Title'], fontSize=16, spaceAfter=6)
s_sub = ParagraphStyle('S', parent=styles['Normal'], fontSize=9, textColor=colors.grey, spaceAfter=12)
s_h1 = ParagraphStyle('H1', parent=styles['Heading1'], fontSize=12, textColor=colors.HexColor('#1a1a2e'), spaceAfter=4)
s_h2 = ParagraphStyle('H2', parent=styles['Heading2'], fontSize=10, textColor=colors.HexColor('#2c5282'), spaceAfter=3)
s_body = ParagraphStyle('B', parent=styles['Normal'], fontSize=9, leading=14, spaceAfter=4)

REC_COLORS = {
    'Ready to Submit': colors.HexColor('#276749'),
    'Validate First': colors.HexColor('#744210'),
    'Do Not Submit': colors.HexColor('#742a2a')
}

def safe(text):
    return str(text).replace('&','&amp;').replace('<','&lt;').replace('>','&gt;')

story = []
story.append(Spacer(1, 8*mm))
story.append(Paragraph('Candidate Matching & Recommendation Report', s_title))
story.append(Paragraph(
    f"Generated: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M')} | Candidates: {len(results)}",
    s_sub))
story.append(HRFlowable(width='100%', thickness=1, color=colors.HexColor('#e0e0e0')))
story.append(Spacer(1, 6*mm))

# Recommendation table
story.append(Paragraph('Recommendation Summary', s_h1))
story.append(Spacer(1, 3*mm))
s_header = ParagraphStyle('H', parent=styles['Normal'], fontSize=8, textColor=colors.white, fontName='Helvetica-Bold')
table_data = [[
    Paragraph('Candidate', s_header),
    Paragraph('Recommendation', s_header),
    Paragraph('Brief Reason', s_header),
    Paragraph('Main Risk', s_header)
]]
for r in results:
    table_data.append([
        Paragraph(safe(r['name']), s_body),
        Paragraph(safe(r['recommendation']), s_body),
        Paragraph(safe(r['brief_reason']), s_body),
        Paragraph(safe(r['main_risk']), s_body)
    ])
t = Table(table_data, colWidths=[30*mm, 28*mm, 55*mm, 55*mm])
t.setStyle(TableStyle([
    ('BACKGROUND', (0,0), (-1,0), colors.HexColor('#2c5282')),
    ('TEXTCOLOR', (0,0), (-1,0), colors.white),
    ('FONTSIZE', (0,0), (-1,-1), 8),
    ('PADDING', (0,0), (-1,-1), 5),
    ('ROWBACKGROUNDS', (0,1), (-1,-1), [colors.white, colors.HexColor('#f7fafc')]),
    ('GRID', (0,0), (-1,-1), 0.5, colors.HexColor('#e0e0e0')),
    ('VALIGN', (0,0), (-1,-1), 'TOP'),
    ('FONTNAME', (0,0), (-1,0), 'Helvetica-Bold'),
]))
story.append(t)
story.append(Spacer(1, 8*mm))
story.append(HRFlowable(width='100%', thickness=1, color=colors.HexColor('#e0e0e0')))
story.append(Spacer(1, 6*mm))

# Detailed reports
story.append(Paragraph('Detailed Candidate Reports', s_h1))
story.append(Spacer(1, 4*mm))

for r in results:
    rec_color = REC_COLORS.get(r['recommendation'], colors.grey)
    s_rec = ParagraphStyle('R', parent=styles['Normal'], fontSize=10,
        textColor=rec_color, spaceBefore=2, spaceAfter=4, fontName='Helvetica-Bold')

    story.append(Paragraph(safe(r['name']), s_h1))
    story.append(Paragraph(safe(r['recommendation']), s_rec))
    story.append(Paragraph(f"Reason: {safe(r['brief_reason'])}", s_body))
    story.append(Paragraph(f"Main Risk: {safe(r['main_risk'])}", s_body))

    story.append(Spacer(1, 2*mm))
    story.append(Paragraph('Strengths', s_h2))
    for s in r['strengths']:
        story.append(Paragraph(f'• {safe(s)}', s_body))

    story.append(Spacer(1, 2*mm))
    story.append(Paragraph('Risks', s_h2))
    for s in r['risks']:
        story.append(Paragraph(f'• {safe(s)}', s_body))

    if r['validation_questions']:
        story.append(Spacer(1, 2*mm))
        story.append(Paragraph('Recruiter Validation Questions', s_h2))
        for i, q in enumerate(r['validation_questions'], 1):
            story.append(Paragraph(f'{i}. {safe(q)}', s_body))

    if r['client_summary']:
        story.append(Spacer(1, 2*mm))
        story.append(Paragraph('Client-Ready Summary', s_h2))
        story.append(Paragraph(safe(r['client_summary']), s_body))

    story.append(Spacer(1, 4*mm))
    story.append(HRFlowable(width='100%', thickness=0.5, color=colors.HexColor('#e0e0e0')))
    story.append(Spacer(1, 4*mm))

doc.build(story)
print(f'Report exported: {output_filename}')
files.download(output_filename)


Report exported: ocbridge_candidate_report.pdf


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>